In [1]:
import plotly.express as px
import torch

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=4,
    mode_std=0.1,
    data_scale=1.0,
    offset=1.0,
    ambient_dimension=8,
)
data_config.visualize()

In [4]:
samples_per_mode = 300
mode_colors = ["#264653", "#2a9d8f", "#e76f51", "#e9c46a"]

samples_hd = torch.cat(
    [
        data_config.sample_class(mode_id=mode_id, batch_size=samples_per_mode)
        for mode_id in range(data_config.num_classes)
    ],
    dim=0,
)
samples_2d = data_config.project(samples_hd).cpu()
mode_ids = torch.repeat_interleave(
    torch.arange(data_config.num_classes),
    repeats=samples_per_mode,
).cpu()
mode_labels = [f"mode {mode_id}" for mode_id in mode_ids.tolist()]

centers_2d = data_config.mode_centers_2d().cpu()

In [5]:
fig = px.scatter(
    x=samples_2d[:, 0].numpy(),
    y=samples_2d[:, 1].numpy(),
    color=mode_labels,
    color_discrete_sequence=mode_colors,
    title="Conditional samples from a 4-mode Gaussian mixture",
    labels={"x": "projected x", "y": "projected y", "color": "mode"},
)
fig.update_traces(marker={"size": 7, "opacity": 0.60})
fig.add_scatter(
    x=centers_2d[:, 0].numpy(),
    y=centers_2d[:, 1].numpy(),
    mode="markers",
    marker={"size": 14, "symbol": "x", "color": "black", "line": {"width": 1}},
    name="mode centers",
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(width=700, height=650)
fig.show()
